<a href="https://colab.research.google.com/github/jgromero/drl-csic/blob/main/code/stable-baselines/cliffwalking-sb3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DQN for _cliff walking_ with Stable-Baselines3

[**Juan Gómez Romero**](https://ccia.ugr.es/~jgomez)  
Departamento de Ciencias de la Computación e Inteligencia Artificial  
Universidad de Granada  
This work is licensed under the [Apache License 2.0](https://choosealicense.com/licenses/apache-2.0/).

---

Example based on:
> R.S. Sutton, A.G. Barto (2018) Reinforcement Learning. Chapters 6.5: Q-learning: Off-policy TD Control, 6.6: Expected Sarsa.

> A. Raffin et al. (2021) Stable-Baselines3: Reliable Reinforcement Learning Implementations. _Journal of Machine Learning Research_ 22(268):1-8. Available on [GitHub](https://github.com/DLR-RM/stable-baselines3).

_Consider the gridworld shown to the right. This is a standard undiscounted, episodic task, with start and goal states, and the usual actions causing movement up, down, right, and left. Reward is -1 on all transitions except those into the region marked "The Cliff". Stepping into this region incurs a reward of -100 and sends the agent instantly back to the start._

<img src="https://github.com/jgromero/drl-csic/blob/main/img/cliffwalking.png?raw=true" width=500/>

In [cliffwalking-qlearning.ipynb](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/cliffwalking-qlearning.ipynb) we solved this problem with our own implementation of tabular Q-Learning. Here we use the [DQN](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html) implementation of [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) (SB3), a library of reliable, tested implementations of the most common DRL algorithms built on PyTorch.

SB3 does not include tabular methods: DQN replaces the $Q$ table with a neural network that approximates $Q(s, a)$. This is not necessary for a problem with only 48 states, but it shows that the same code can be applied to problems with much larger (or continuous) state spaces, where a table is not feasible.

## Environment

In [ ]:
%pip install "gymnasium>=1.1" "stable-baselines3>=2.6" tqdm rich

[Cliff Walking](https://gymnasium.farama.org/environments/toy_text/cliff_walking/) is included in [Gymnasium](https://gymnasium.farama.org/) as a sample (and simple) environment.

<img src="https://gymnasium.farama.org/_images/cliff_walking.gif"/>

The observation is the index of the cell where the agent is ($0$ to $47$). SB3 automatically encodes this discrete observation as a _one-hot_ vector of size $48$ before feeding it to the Q-network.

Falling into the cliff does not end the episode, so a bad policy can walk forever. To prevent this, we limit the length of the episodes to 100 steps with the `max_episode_steps` parameter. We also wrap the environment with a [`Monitor`](https://stable-baselines3.readthedocs.io/en/master/common/monitor.html) to keep track of the score of each training episode.

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

env = Monitor(gym.make("CliffWalking-v1", max_episode_steps=100))
print(env.observation_space)
print(env.action_space)

## DQN algorithm

The [`DQN`](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html) class includes the Q-network, the target network, the replay buffer and the $\epsilon$-greedy exploration strategy. Instead of a number of episodes, SB3 trains for a given number of time steps (`total_timesteps`).

In [ ]:
model = DQN(
    "MlpPolicy",                        # Q-network: multilayer perceptron
    env,
    learning_rate=1e-3,
    batch_size=64,
    buffer_size=50_000,
    learning_starts=1000,               # steps of random actions before learning starts
    gamma=0.99,
    target_update_interval=500,         # copy the Q-network into the target network every 500 steps
    train_freq=4,                       # update the Q-network every 4 steps
    exploration_fraction=0.3,           # epsilon decays linearly during the first 30% of training...
    exploration_final_eps=0.05,         # ...to this value
    policy_kwargs=dict(net_arch=[64, 64]),
    seed=0,
    verbose=0,
)

model.learn(total_timesteps=50_000, progress_bar=True)
model.save("cliffwalking-sb3-dqn")   # save trained agent

# plot the scores
plot_every = 20
scores = env.get_episode_rewards()
avg_scores = [np.mean(scores[i:i+plot_every]) for i in range(0, len(scores), plot_every)]
plt.plot(np.arange(0, len(scores), plot_every), avg_scores)
plt.xlabel('Episode number')
plt.ylabel('Average score of the last %d episodes' % plot_every)
plt.show()

The optimal policy (walking along the edge of the cliff) obtains a score of $-13$. Let us evaluate the agent with the greedy policy (`deterministic=True`). Since the environment is deterministic, a single episode is enough.

In [ ]:
mean_score, _ = evaluate_policy(model, Monitor(gym.make("CliffWalking-v1", max_episode_steps=100)), n_eval_episodes=1, deterministic=True)
print('Score of the greedy policy:', mean_score)

The resulting policy is the following. We obtain it by asking the model for the greedy action (`predict`) in each of the 48 states.

In [ ]:
states = np.arange(48)
actions, _ = model.predict(states, deterministic=True)

policy_plot = actions.reshape(4, 12)
policy_plot[3, 1:] = -1   # cliff and goal cells are never visited
print("\nEstimated optimal policy (UP = 0, RIGHT = 1, DOWN = 2, LEFT = 3, N/A = -1):")
print(policy_plot)

We can visualize the values of the state-value function $v_\pi$ for the estimated policy $\pi$ and compare them with the optimal policy values $v_{\pi_*}$. Here, the values $Q(s, a)$ are not read from a table, but computed by the Q-network (`model.q_net`).

Note that we trained with $\gamma = 0.99$ instead of $\gamma = 1$, so the values of the states far from the goal are slightly higher (less negative) than the optimal ones.

(The `plot_values()` function is required for the visualization.)

In [ ]:
def plot_values(V):
    # reshape the state-value function
    V = np.reshape(V, (4,12))
    # plot the state-value function
    fig = plt.figure(figsize=(15,5))
    ax = fig.add_subplot(111)
    im = ax.imshow(V, cmap='cool')
    for (j,i),label in np.ndenumerate(V):
        ax.text(i, j, np.round(label,3), ha='center', va='center', fontsize=14)
    plt.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
    plt.title('State-Value Function')
    plt.show()

__State-value function for the agent's learnt policy $v_\pi$__

In [ ]:
import torch

obs, _ = model.policy.obs_to_tensor(states)   # one-hot encoding of the 48 states
with torch.no_grad():
    Q = model.q_net(obs).cpu().numpy()        # Q values of the 48 states, shape (48, 4)

V = Q.max(axis=1)
V[37:] = 0   # cliff and goal cells are never visited
plot_values(V)

__State-value function for the optimal policy  $v_{\pi_*}$__

In [ ]:
# The state-values for the optimal policy can be manually calculated
V_opt = np.zeros((4,12))
V_opt[0][:] = -np.arange(3, 15)[::-1]
V_opt[1][:] = -np.arange(3, 15)[::-1] + 1
V_opt[2][:] = -np.arange(3, 15)[::-1] + 2
V_opt[3][0] = -13
plot_values(V_opt)